# ASDE — Cloud Bootstrap v0.1

**Purpose:** initialise and verify the cloud foundation for the Autonomous Scientific Discovery Engine.

This notebook:
- Runs in Google Colab (cloud runtime).
- Connects to a Neon PostgreSQL database.
- Creates the initial ASDE metadata schema.
- Checks Groq/Gemini model listings without making inference calls.
- Clones the private GitHub repository and pushes a sanitised health report.
- Does **not** mount Google Drive.
- Does **not** launch an autonomous agent swarm yet.

## Important constraints

- Colab free runtimes are temporary; resources are not guaranteed.
- Use this notebook for interactive, bounded research jobs—not as an always-on server or distributed worker cluster.
- Database state persists in Neon; the Colab VM filesystem is temporary.
- Never commit API keys, database URLs, tokens, private data or unpublished raw datasets.
- Model-list visibility does not prove free inference entitlement or scientific quality.

## Step 0 — Create the cloud resources

### A. GitHub repository

1. Open [GitHub New Repository](https://github.com/new).
2. Create a repository named **ASDE**.
3. Set visibility to **Private** (recommended for unpublished research).
4. Initialise with a README.
5. Add a Python `.gitignore`.
6. Use `main` as the default branch.

Create a **fine-grained personal access token**:
- Repository access: only `ASDE`.
- Repository permissions: **Contents: Read and write**.
- Metadata: read-only (default).
- Set an expiry date.

### B. Neon PostgreSQL

1. Open [Neon](https://neon.tech/) and create a free project named `asde-core`.
2. Choose the nearest available region.
3. Keep the default Postgres database and role initially.
4. Open **Connect** and copy the PostgreSQL connection string.
5. Use the direct connection string and ensure SSL is enabled (`sslmode=require`).

### C. Colab Secrets

1. Open [Google Colab](https://colab.research.google.com/).
2. Open this notebook from GitHub after authorising GitHub access, or upload it.
3. Open the **Secrets** panel (key icon).
4. Add these secrets and enable notebook access:

Required:
- `GITHUB_REPO` — `your-github-username/ASDE`
- `GITHUB_TOKEN` — your fine-grained token
- `NEON_DATABASE_URL` — Neon PostgreSQL connection string

Optional model keys:
- `GROQ_API_KEY`
- `GEMINI_API_KEY`

Recommended artifact storage (no Google Drive):
1. In the Neon project, open **Connect → Storage** and use the S3 client/Python instructions.
2. Create a private bucket named `asde-artifacts` in the Neon Storage interface if it is not already present.
3. Create a credential with `storage:write` scope (which includes read access).
4. Add these Colab Secrets from the Neon-provided connection details:
   - `NEON_S3_ENDPOINT`
   - `NEON_S3_ACCESS_KEY`
   - `NEON_S3_SECRET_KEY`
   - `NEON_S3_REGION`
   - `NEON_S3_BUCKET` — `asde-artifacts`

Never paste secrets into notebook cells or commit them.

## Step 1 — Install lightweight client dependencies

No local language model is downloaded.

In [ ]:
%pip -q install "psycopg[binary]>=3.2,<4" requests psutil boto3

## Step 2 — Read Colab Secrets securely

Secret values are never printed.

In [ ]:
from google.colab import userdata
import os, re, json, time, uuid, hashlib, platform, subprocess, base64
from pathlib import Path
from datetime import datetime, timezone

def read_secret(name, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        return value.strip()
    if required:
        raise RuntimeError(
            f"Missing Colab Secret: {name}. Add it in the Secrets panel and enable notebook access."
        )
    return None

GITHUB_REPO = read_secret("GITHUB_REPO", required=True)
GITHUB_TOKEN = read_secret("GITHUB_TOKEN", required=True)
NEON_DATABASE_URL = read_secret("NEON_DATABASE_URL", required=True)
GROQ_API_KEY = read_secret("GROQ_API_KEY")
GEMINI_API_KEY = read_secret("GEMINI_API_KEY")
NEON_S3_ENDPOINT = read_secret("NEON_S3_ENDPOINT")
NEON_S3_ACCESS_KEY = read_secret("NEON_S3_ACCESS_KEY")
NEON_S3_SECRET_KEY = read_secret("NEON_S3_SECRET_KEY")
NEON_S3_REGION = read_secret("NEON_S3_REGION")
NEON_S3_BUCKET = read_secret("NEON_S3_BUCKET")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPO):
    raise ValueError("GITHUB_REPO must look like owner/ASDE.")

print({
    "github_repo_configured": bool(GITHUB_REPO),
    "github_token_configured": bool(GITHUB_TOKEN),
    "neon_database_configured": bool(NEON_DATABASE_URL),
    "groq_key_configured": bool(GROQ_API_KEY),
    "gemini_key_configured": bool(GEMINI_API_KEY),
    "neon_object_storage_configured": all([NEON_S3_ENDPOINT, NEON_S3_ACCESS_KEY, NEON_S3_SECRET_KEY, NEON_S3_REGION, NEON_S3_BUCKET]),
})

## Step 3 — Inspect the temporary Colab runtime

GPU and RAM availability vary by session. No particular hardware is assumed.

In [ ]:
import psutil

ram = psutil.virtual_memory()
gpu_info = "No GPU detected"
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0 and result.stdout.strip():
        gpu_info = result.stdout.strip()
except Exception:
    pass

runtime_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "ram_total_gb": round(ram.total / (1024**3), 2),
    "ram_available_gb": round(ram.available / (1024**3), 2),
    "gpu": gpu_info,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
}
print(json.dumps(runtime_info, indent=2))

## Step 4 — Connect to Neon and initialise the metadata schema

Large datasets and binary simulation outputs should be stored as external artifacts, not database blobs.

In [ ]:
import psycopg
from psycopg.rows import dict_row

conn = psycopg.connect(
    NEON_DATABASE_URL,
    connect_timeout=20,
    row_factory=dict_row,
)
conn.execute("SELECT 1 AS ok").fetchone()

schema_sql = [
    """CREATE TABLE IF NOT EXISTS asde_schema_version (
        version TEXT PRIMARY KEY,
        applied_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )""",
    """CREATE TABLE IF NOT EXISTS asde_projects (
        project_id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        description TEXT,
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        metadata JSONB NOT NULL DEFAULT '{}'::jsonb
    )""",
    """CREATE TABLE IF NOT EXISTS asde_runs (
        run_id TEXT PRIMARY KEY,
        project_id TEXT NOT NULL REFERENCES asde_projects(project_id),
        mission TEXT NOT NULL,
        status TEXT NOT NULL DEFAULT 'created',
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        metadata JSONB NOT NULL DEFAULT '{}'::jsonb
    )""",
    """CREATE TABLE IF NOT EXISTS asde_agents (
        agent_id TEXT PRIMARY KEY,
        run_id TEXT NOT NULL REFERENCES asde_runs(run_id) ON DELETE CASCADE,
        role TEXT NOT NULL,
        provider TEXT,
        model_id TEXT,
        status TEXT NOT NULL DEFAULT 'registered',
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        metadata JSONB NOT NULL DEFAULT '{}'::jsonb
    )""",
    """CREATE TABLE IF NOT EXISTS asde_messages (
        message_id TEXT PRIMARY KEY,
        run_id TEXT NOT NULL REFERENCES asde_runs(run_id) ON DELETE CASCADE,
        sender_agent_id TEXT,
        recipient_agent_id TEXT,
        topic TEXT NOT NULL,
        payload JSONB NOT NULL,
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )""",
    """CREATE TABLE IF NOT EXISTS asde_hypotheses (
        hypothesis_id TEXT PRIMARY KEY,
        run_id TEXT NOT NULL REFERENCES asde_runs(run_id) ON DELETE CASCADE,
        parent_hypothesis_id TEXT,
        title TEXT NOT NULL,
        statement TEXT NOT NULL,
        status TEXT NOT NULL DEFAULT 'candidate',
        verification_status TEXT NOT NULL DEFAULT 'unverified',
        evidence JSONB NOT NULL DEFAULT '[]'::jsonb,
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        metadata JSONB NOT NULL DEFAULT '{}'::jsonb
    )""",
    """CREATE TABLE IF NOT EXISTS asde_artifacts (
        artifact_id TEXT PRIMARY KEY,
        run_id TEXT NOT NULL REFERENCES asde_runs(run_id) ON DELETE CASCADE,
        artifact_type TEXT NOT NULL,
        uri TEXT NOT NULL,
        sha256 TEXT,
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        metadata JSONB NOT NULL DEFAULT '{}'::jsonb
    )""",
    """CREATE TABLE IF NOT EXISTS asde_events (
        event_id BIGSERIAL PRIMARY KEY,
        run_id TEXT REFERENCES asde_runs(run_id) ON DELETE CASCADE,
        event_type TEXT NOT NULL,
        payload JSONB NOT NULL DEFAULT '{}'::jsonb,
        created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )""",
    "CREATE INDEX IF NOT EXISTS idx_asde_runs_project ON asde_runs(project_id, created_at DESC)",
    "CREATE INDEX IF NOT EXISTS idx_asde_agents_run ON asde_agents(run_id)",
    "CREATE INDEX IF NOT EXISTS idx_asde_messages_run_time ON asde_messages(run_id, created_at)",
    "CREATE INDEX IF NOT EXISTS idx_asde_messages_recipient ON asde_messages(recipient_agent_id, created_at)",
    "CREATE INDEX IF NOT EXISTS idx_asde_hypotheses_run_status ON asde_hypotheses(run_id, status)",
    "CREATE INDEX IF NOT EXISTS idx_asde_events_run_time ON asde_events(run_id, created_at)",
]

with conn.transaction():
    for statement in schema_sql:
        conn.execute(statement)
    conn.execute(
        "INSERT INTO asde_schema_version(version) VALUES (%s) ON CONFLICT DO NOTHING",
        ("0.1.0",)
    )

print("Neon connection successful; ASDE schema v0.1.0 is ready.")

## Step 5 — Verify database persistence

This writes and reads one small test event. It does not create a research mission or call an LLM.

In [ ]:
smoke_id = str(uuid.uuid4())
with conn.transaction():
    conn.execute(
        """
        INSERT INTO asde_events(event_type, payload)
        VALUES (%s, %s::jsonb)
        """,
        ("bootstrap_smoke_test", json.dumps({
            "smoke_id": smoke_id,
            "schema_version": "0.1.0",
            "timestamp_utc": datetime.now(timezone.utc).isoformat()
        }))
    )

row = conn.execute(
    """
    SELECT event_id, event_type, payload, created_at
    FROM asde_events
    WHERE event_type = 'bootstrap_smoke_test'
      AND payload->>'smoke_id' = %s
    ORDER BY event_id DESC
    LIMIT 1
    """,
    (smoke_id,)
).fetchone()

assert row is not None, "Database read-after-write test failed."
print("Database persistence test: PASS")
print({"event_id": row["event_id"], "event_type": row["event_type"]})

## Step 6 — Discover provider model IDs without inference calls

This uses model-list endpoints only. It does **not** generate text, benchmark reasoning, or establish free-tier entitlement.

In [ ]:
import requests

provider_report = {
    "groq": {"key_configured": bool(GROQ_API_KEY), "model_list_reachable": False, "model_ids": []},
    "gemini": {"key_configured": bool(GEMINI_API_KEY), "model_list_reachable": False, "model_ids": []},
}

if GROQ_API_KEY:
    try:
        response = requests.get(
            "https://api.groq.com/openai/v1/models",
            headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
            timeout=20,
        )
        if response.ok:
            provider_report["groq"]["model_list_reachable"] = True
            provider_report["groq"]["model_ids"] = sorted(
                item.get("id", "") for item in response.json().get("data", [])
                if item.get("id")
            )
        else:
            provider_report["groq"]["http_status"] = response.status_code
    except Exception as exc:
        provider_report["groq"]["error_type"] = type(exc).__name__

if GEMINI_API_KEY:
    try:
        response = requests.get(
            "https://generativelanguage.googleapis.com/v1beta/models",
            params={"key": GEMINI_API_KEY},
            timeout=20,
        )
        if response.ok:
            provider_report["gemini"]["model_list_reachable"] = True
            provider_report["gemini"]["model_ids"] = sorted(
                item.get("name", "") for item in response.json().get("models", [])
                if item.get("name")
            )
        else:
            provider_report["gemini"]["http_status"] = response.status_code
    except Exception as exc:
        provider_report["gemini"]["error_type"] = type(exc).__name__

safe_provider_report = {
    name: {
        "key_configured": details["key_configured"],
        "model_list_reachable": details["model_list_reachable"],
        "model_count": len(details["model_ids"]),
        "model_ids": details["model_ids"],
        "http_status": details.get("http_status"),
        "error_type": details.get("error_type"),
    }
    for name, details in provider_report.items()
}
print(json.dumps(safe_provider_report, indent=2))
print("\\nA listed model is not automatically free, benchmark-qualified, or included in your account's inference quota.")

## Step 7 — Clone the private GitHub repository and push a sanitised health report

The token is supplied through a temporary Git process environment header. It is not embedded in the remote URL, printed, or written into the repository.

The health report contains runtime metadata and provider configuration booleans only. It never contains credentials or database connection strings.

In [ ]:
from pathlib import Path
import subprocess
import base64

repo_dir = Path("/content/ASDE")
repo_url = f"https://github.com/{GITHUB_REPO}.git"

auth_value = base64.b64encode(
    f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
).decode("ascii")

git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraheader"
git_env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth_value}"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
        env=git_env,
        capture_output=True,
        text=True,
    )

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=repo_dir,
        env=git_env,
        check=True,
        capture_output=True,
        text=True,
    )

git("config", "user.name", "ASDE Bootstrap")
git("config", "user.email", "asde-bootstrap@users.noreply.github.com")

current_branch = git("branch", "--show-current").stdout.strip()
if current_branch != "main":
    git("checkout", "-B", "main")

health_report = {
    "asde_bootstrap_version": "0.1.0",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "runtime": runtime_info,
    "database": {
        "connected": True,
        "schema_version": "0.1.0",
        "read_after_write": True
    },
    "providers": {
        name: {
            "key_configured": details["key_configured"],
            "model_list_reachable": details["model_list_reachable"],
            "model_count": len(details["model_ids"])
        }
        for name, details in provider_report.items()
    },
    "notes": [
        "No API keys or database credentials are stored in this report.",
        "Model listing does not establish free inference entitlement or benchmark qualification."
    ]
}

health_dir = repo_dir / "bootstrap"
health_dir.mkdir(parents=True, exist_ok=True)
health_path = health_dir / "healthcheck.json"
health_path.write_text(json.dumps(health_report, indent=2), encoding="utf-8")

git("add", "bootstrap/healthcheck.json")
status = git("status", "--porcelain").stdout.strip()

if status:
    git("commit", "-m", "Add ASDE cloud bootstrap health report")
    git("push", "origin", "HEAD:main")
    print("Sanitised health report pushed to GitHub.")
else:
    print("No GitHub changes to commit.")

print("Repository:", GITHUB_REPO)
print("Branch: main")
print("Health report: bootstrap/healthcheck.json")

## Step 8 — Verify Neon Object Storage (optional but recommended)

This is the persistent artifact store for users who do not want Google Drive.
If the five `NEON_S3_*` secrets are not configured, the cell safely skips the upload.

In [ ]:
object_storage_status = {"configured": False, "upload_verified": False}

if all([NEON_S3_ENDPOINT, NEON_S3_ACCESS_KEY, NEON_S3_SECRET_KEY, NEON_S3_REGION, NEON_S3_BUCKET]):
    import boto3
    from botocore.config import Config

    s3 = boto3.client(
        "s3",
        endpoint_url=NEON_S3_ENDPOINT,
        region_name=NEON_S3_REGION,
        aws_access_key_id=NEON_S3_ACCESS_KEY,
        aws_secret_access_key=NEON_S3_SECRET_KEY,
        config=Config(s3={"addressing_style": "path"}),
    )

    object_key = "bootstrap/healthcheck.json"
    s3.put_object(
        Bucket=NEON_S3_BUCKET,
        Key=object_key,
        Body=json.dumps(health_report, indent=2).encode("utf-8"),
        ContentType="application/json",
    )
    s3.head_object(Bucket=NEON_S3_BUCKET, Key=object_key)

    object_storage_status = {
        "configured": True,
        "upload_verified": True,
        "bucket": NEON_S3_BUCKET,
        "object_key": object_key,
    }
    print("Neon Object Storage upload: PASS")
    print(object_storage_status)
else:
    print("Neon Object Storage not configured; skipped. No Google Drive is used.")
    print("Add the five NEON_S3_* Colab Secrets to enable persistent artifact uploads.")

In [ ]:
final_status = {
    "github_repository_cloned": repo_dir.exists(),
    "neon_connected": True,
    "database_schema": "0.1.0",
    "groq_key_configured": bool(GROQ_API_KEY),
    "gemini_key_configured": bool(GEMINI_API_KEY),
    "google_drive_mounted": False,
    "neon_object_storage_upload_verified": object_storage_status.get("upload_verified", False),
    "llm_generation_calls_made": 0,
    "bootstrap_complete": True,
}
print(json.dumps(final_status, indent=2))

conn.close()
print("\\nASDE Cloud Bootstrap v0.1 completed.")